# Catalog Population

Select an existing catalog below. Its `creation-settings.json` snapshot is authoritative for population; this notebook does not read the editable `configs/params-config.json` creation configuration. See the [rerun runbook](../docs/source/lwi_geometry_rerun.rst). Run discovery separately for 24, 48, and 72 hours.


In [ ]:
from datetime import datetime
from pathlib import Path

from stormhub.logger import initialize_logger
from stormhub.met.catalog_setup import load_catalog_settings
from stormhub.met.storm_catalog import (
    StormCatalog,
    add_storm_dss_files,
    create_normal_precip,
    new_collection,
    resume_collection,
    stac_to_parquet,
)
from stormhub.met.zarr_to_dss import NOAADataVariable

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

# Select the catalog to populate independently of the next catalog's creation config.
CATALOG_ID = "lwi-region3-geometry-v2"
CATALOG_DIR = REPO_ROOT / "catalogs" / CATALOG_ID
CATALOG_FILE = CATALOG_DIR / "catalog.json"
settings = load_catalog_settings(CATALOG_DIR / "creation-settings.json")
if settings["catalog_id"] != CATALOG_ID:
    raise ValueError("The creation-settings snapshot does not belong to the selected catalog.")
run_settings = settings["params"]

CATALOG_FILE


## Population Parameters

`SMOKE_TEST = True` limits the run to the dates in `SPECIFIC_DATES`. Set it to `False` for the full configured date range.

In [ ]:
# All Collection Args
START_DATE = run_settings["start_date"] # start of date span used to generate candidate AORC events, AORC data starts "1979-02-01"
END_DATE = run_settings["end_date"] # end of date span used to generate candidate AORC events
TOP_N_EVENTS = run_settings["top_n_events"] # total number of ranked storm events retained as STAC items, e.g. 100 for date range 2016-2025 represents the top 10 events for 10 years

# Collection Args
STORM_DURATION_HOURS = run_settings["storm_duration_hours"] # rolling accumulation window, e.g. 48hr-events
MIN_PRECIP_THRESHOLD = run_settings["min_precip_threshold_inches"] # precipitation cutoff before ranking
CHECK_EVERY_N_HOURS = run_settings["check_every_n_hours"] # search interval; 6 is denser and would check the totals every 6 hours, or 4 times a day)
NUM_WORKERS = run_settings["num_workers"] # number of parallel workers for event processing
USE_THREADS = run_settings["use_threads"]  # Windows: keep this False.

# Development controls
SMOKE_TEST = False # limits events to specific dates quick testing
SPECIFIC_DATES = [datetime(2010, 1, 1, 0), datetime(2020, 12, 31, 0)] # targeted dates for smoke tests or reruns. this is buggy, currently only limits to the specific dates entered and not a range
WITH_TRACEBACK = True # detailed debugging output
CREATE_NEW_ITEMS = True # writes storm item JSON when true; updates existing metadata when false

# Optional derived products
RUN_DSS_EXPORT = False # adds DSS files to storm items
DSS_OUTPUT_MODES = tuple(run_settings["dss_output_modes"]) # source preserves event location; target moves it to the watershed
TARGET_BUFFER_KM = run_settings["target_buffer_km"] # target DSS extent beyond the watershed boundary
DSS_ITEM_IDS = ["1"] # set to a list such as ["1"] for a targeted export
RUN_NORMAL_PRECIP = False # builds annual max/normal precip grids
RUN_GEOPARQUET_EXPORT = True # writes EDA-friendly parquet table

params = {
    "start_date": START_DATE,
    "end_date": END_DATE,
    "top_n_events": TOP_N_EVENTS,
    "storm_duration_hours": STORM_DURATION_HOURS,
    "min_precip_threshold": MIN_PRECIP_THRESHOLD,
    "check_every_n_hours": CHECK_EVERY_N_HOURS,
    "num_workers": NUM_WORKERS,
    "use_threads": USE_THREADS,
    "smoke_test": SMOKE_TEST,
    "dss_output_modes": DSS_OUTPUT_MODES,
    "target_buffer_km": TARGET_BUFFER_KM,
    "dss_item_ids": DSS_ITEM_IDS,
}
params

## Inputs
| Input | Purpose |
| --- | --- |
| `START_DATE`, `END_DATE` | Inclusive period used to generate candidate AORC storm start times. |
| `TOP_N_EVENTS` | Number of ranked storms retained as STAC items after filtering and ranking. |
| `STORM_DURATION_HOURS` | Rolling precipitation accumulation window. Also determines the collection id, such as `48hr-events`. |
| `MIN_PRECIP_THRESHOLD` | Minimum event precipitation threshold used before ranking storms. |
| `CHECK_EVERY_N_HOURS` | Candidate start-time interval. Smaller values search more densely and cost more. |
| `NUM_WORKERS` | Parallel workers used while collecting event stats and creating items. |
| `USE_THREADS` | Executor mode. Use `False` on native Windows per the StormHub guidance. |
| `SPECIFIC_DATES` | Optional list of exact candidate start datetimes. Useful for smoke tests or targeted reruns. |
| `CREATE_NEW_ITEMS` | If `True`, writes item JSON for selected storms; if `False`, updates ranking metadata on existing items. |
| `WITH_TRACEBACK` | Includes tracebacks in error logging for easier debugging. |
| `RUN_DSS_EXPORT` | Adds DSS assets to storm items. This can be expensive and requires `hecdss`. |
| `DSS_OUTPUT_MODES` | Selects source-location DSS, target-transposed DSS, or both. |
| `TARGET_BUFFER_KM` | Buffer around the modeling watershed retained in target DSS grids. |
| `DSS_ITEM_IDS` | Optional event item IDs for targeted DSS generation or reruns. |
| `RUN_NORMAL_PRECIP` | Builds an annual-max/normal precipitation GeoTIFF. This is also a large AORC workload. |
| `RUN_GEOPARQUET_EXPORT` | Converts populated STAC items to GeoParquet for easier analysis and sharing. |

## Load Catalog

In [ ]:
if not CATALOG_FILE.exists():
    raise FileNotFoundError(f"Create the base catalog first: {CATALOG_FILE}")

initialize_logger()
storm_catalog = StormCatalog.from_file(str(CATALOG_FILE))
if storm_catalog.id != CATALOG_ID:
    raise ValueError("The STAC catalog ID does not match the selected catalog and its creation settings.")
storm_catalog.id, storm_catalog.spm.catalog_file


## Create Or Update Event Collection

For the full run, set `SMOKE_TEST = False` in the parameter cell and rerun from there.

In [ ]:
specific_dates = SPECIFIC_DATES if SMOKE_TEST else None

storm_collection = new_collection(
    storm_catalog,
    start_date=START_DATE,
    end_date=END_DATE,
    storm_duration=STORM_DURATION_HOURS,
    min_precip_threshold=MIN_PRECIP_THRESHOLD,
    top_n_events=TOP_N_EVENTS,
    check_every_n_hours=CHECK_EVERY_N_HOURS,
    specific_dates=specific_dates,
    use_threads=USE_THREADS,
    num_workers=NUM_WORKERS,
    with_tb=WITH_TRACEBACK,
    create_new_items=CREATE_NEW_ITEMS,
)

storm_collection.id if storm_collection else None

## Resume missing dates (alternative to discovery)

Run this cell only to continue an interrupted search with unchanged geometry and settings. Do not run both discovery and resume as routine sequential steps. Reconcile ranks/Items before export if resumption changes the ranked population.


In [ ]:
RUN_RESUME = False  # Enable only for an interrupted search.
if RUN_RESUME:
    resume_collection(
         catalog=str(CATALOG_FILE),
         start_date=START_DATE,
         end_date=END_DATE,
         storm_duration=STORM_DURATION_HOURS,
         min_precip_threshold=MIN_PRECIP_THRESHOLD,
         top_n_events=TOP_N_EVENTS,
         check_every_n_hours=CHECK_EVERY_N_HOURS,
         num_workers=NUM_WORKERS,
         with_tb=WITH_TRACEBACK,
         create_items=CREATE_NEW_ITEMS,
         use_threads=USE_THREADS,
     )


## Optional DSS assets

Export is off by default. After reviewing the population, enable `RUN_DSS_EXPORT` and smoke-test `DSS_ITEM_IDS=["1"]` for each duration. Select explicit remaining IDs for the full export; `None` regenerates all selected outputs, including successful smoke files.


In [ ]:
dss_export_result = None

if RUN_DSS_EXPORT:
    dss_export_result = add_storm_dss_files(
        storm_catalog,
        collection_id=f"{STORM_DURATION_HOURS}hr-events",
        aoi_name=CATALOG_ID,
        use_valid_region=run_settings["use_valid_region"],
        variable_duration_map={NOAADataVariable.APCP: STORM_DURATION_HOURS},
        dss_output_dir=str(CATALOG_DIR / f"{STORM_DURATION_HOURS}hr-events" / "dss"),
        output_resolution_km=run_settings["output_resolution_km"],
        output_modes=DSS_OUTPUT_MODES,
        target_buffer_km=TARGET_BUFFER_KM,
        item_ids=DSS_ITEM_IDS,
    )
    if dss_export_result["failed_count"]:
        raise RuntimeError(
            f"DSS export failed for {dss_export_result['failed_count']} item(s): "
            f"{dss_export_result['failed_items']}"
        )
else:
    print("RUN_DSS_EXPORT is False; skipping DSS export.")

dss_export_result

### DSS Pipeline Enhancements

#### 1. Define the export API
- Selectable collection_id and specific events.
- Support source and target output modes.
- Configure resolution, variables, buffer distance, and output directory.


#### 2. Reduce output extent
- Ensure we're not exporting the entire transposition region.
- For source DSS, clip to the transposed watershed plus a configurable buffer.
- For target DSS, shift precipitation onto the modeling watershed and clip there.


#### 3. Create stable filenames
- Example: r001_20160308T1800_48h_aorc_shg1k.dss.
- Avoid current date-only filenames.


#### 4. Record HEC metadata
- DSS pathname pattern
- UTC time window
- Record count
- SHG resolution and CRS
- Source and target storm centers
- X/Y transposition offset in model units
- AORC dataset version


#### 5. Validate exports
- Reopen the DSS after writing.
- Confirm 48 hourly precipitation grids for a 48-hour event.
- Verify units, timestamps, grid extent, resolution, missing values, and nonnegative precipitation.
- Compare precipitation totals before and after reprojection.


#### 6. Improve STAC integration
- Use asset keys such as dss-source and dss-target.
- Store relative paths, file size, checksum, validation status, and export configuration.
- Return a failure summary instead of only logging exceptions.


#### 7. Integrate HEC support
- Connect the existing .grid generator for HEC-HMS.
- Keep DSS itself model-neutral so the same files remain usable in HEC-RAS.
- Generate model-specific configuration as a separate derivative.


#### 8. Add focused tests
- DSS write/read round trip
- Exact timestep and pathname checks
- Reprojection and clipping checks
- Source-to-target transposition test
- Small end-to-end STAC export fixture

## Optional Normal Precipitation Grid

This computes annual maximum grids and averages them into a normal precipitation GeoTIFF. Consider testing a short year range before running the full 1980-2024 period.

In [ ]:
if RUN_NORMAL_PRECIP:
    normal_dir = CATALOG_DIR / "normal_precip"
    normal_dir.mkdir(parents=True, exist_ok=True)

    create_normal_precip(
        start_year=1980,
        end_year=2024,
        catalog=storm_catalog,
        storm_duration_hours=STORM_DURATION_HOURS,
        every_n_hours=24,
        months=None,
        ams_zarr_path=str(normal_dir / "ams_grids.zarr"),
        normal_precip_grid_path=str(normal_dir / "normalized_precip.tif"),
    )
else:
    print("RUN_NORMAL_PRECIP is False; skipping normal precipitation grid.")

## Optional GeoParquet Export

GeoParquet gives you a compact table for EDA after the STAC item JSON has been created.

In [ ]:
if RUN_GEOPARQUET_EXPORT and storm_collection is not None:
    parquet_path = CATALOG_DIR / storm_collection.id / "all-items.parquet"
    stac_to_parquet(storm_collection, parquet_file=str(parquet_path))
    print(parquet_path)
else:
    print("Skipping GeoParquet export.")

## Serve The Populated Catalog

Run this from a PowerShell terminal with the `stormhub` environment active:

```powershell
stormhub-server .\catalogs\lwi-region3-geometry-v2 127.0.0.1 5000
```

Then open `http://localhost:5000/catalog.json` or the STAC Browser link shown by the directory listing.